# Kreo Digital — Session 09: MLflow Best Practices & Scaling
**Integrated lab:** From reproducible experiments to a governed model registry.

**Learning goals**
1. Identify runs, parameters, metrics, artifacts, input lineage, and model signatures.
2. Compare real baselines using a leakage-safe split.
3. Track every experiment in MLflow without silently using the test set for selection.
4. Register and promote the best validation model using an alias.
5. Explain why PostgreSQL and object storage are separate MLflow concerns.
6. Design authentication, backups, monitoring, and scaling controls.

This notebook uses **fully synthetic, seeded, fraud-like classification data**, not real customer records or PaySim. No external download is required. Run cells top-to-bottom. The notebook intentionally does not auto-deploy models.

## 1 — Setup and tracking options
Install `pip install -r requirements.txt` in a Python 3.11+ environment.

For a local class, use `sqlite:///mlflow.db` and the default local artifact store. To use the Docker-backed server, run `docker compose up -d --build` in this folder and set `MLFLOW_TRACKING_URI=http://127.0.0.1:5000` **before** starting Jupyter.

**Production warning:** The included Compose deployment is for local learning only. It binds ports to localhost, uses sample credentials, and has no production identity provider or TLS. Never expose it on the internet as-is.

In [ ]:
import os, sys, platform, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, confusion_matrix
from mlflow.tracking import MlflowClient

SEED = 42
uri = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
mlflow.set_tracking_uri(uri)
experiment_name = "kreo-session-09-mlflow-best-practices"
mlflow.set_experiment(experiment_name)
print("Python", platform.python_version(), "MLflow", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

## 2 — Create deterministic data and an auditable split
We generate an imbalanced dataset to mimic a fraud-classification learning exercise. We split into **train, validation, and final test**. Only training data may fit transforms; validation selects the model. Test is evaluated once after selection.

In [ ]:
X, y = make_classification(
    n_samples=8000, n_features=18, n_informative=7, n_redundant=3,
    weights=[0.96, 0.04], flip_y=0.01, class_sep=1.1, random_state=SEED
)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED
)
assert not np.shares_memory(X_train, X_test)
assert len(X_train)+len(X_valid)+len(X_test)==len(X)
counts = {k: {"n":len(v), "fraud_rate":float(np.mean(v))}
          for k,v in [("train",y_train),("validation",y_valid),("test",y_test)]}
print(json.dumps(counts, indent=2))
dataset_fingerprint = hashlib.sha256(
    np.ascontiguousarray(X_train).tobytes()+np.ascontiguousarray(y_train).tobytes()
).hexdigest()
print("Training fingerprint:", dataset_fingerprint)

## 3 — Log two candidate models, including lineage and artifacts
Always record the evaluation dataset, seed, model configuration, Git SHA if available, dependency versions, and a reproducible metric definition. We deliberately disable autologging to teach explicit calls.

We use **PR-AUC (average precision)** as the selection objective because positives are rare. This is not a financial impact metric.

In [ ]:
from mlflow.models import infer_signature
candidates = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, class_weight="balanced", random_state=SEED))
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=120, min_samples_leaf=3, class_weight="balanced",
        random_state=SEED, n_jobs=1
    )
}
results = []
for model_name, estimator in candidates.items():
    with mlflow.start_run(run_name=model_name) as run:
        estimator.fit(X_train, y_train)
        scores = estimator.predict_proba(X_valid)[:,1]
        predictions = (scores >= 0.5).astype(int)
        metrics = {
            "validation_pr_auc":float(average_precision_score(y_valid,scores)),
            "validation_roc_auc":float(roc_auc_score(y_valid,scores)),
            "validation_precision_at_05":float(precision_score(y_valid,predictions,zero_division=0)),
            "validation_recall_at_05":float(recall_score(y_valid,predictions,zero_division=0)),
        }
        mlflow.log_params({
            "model_family":model_name, "random_seed":SEED,
            "train_rows":len(X_train), "validation_rows":len(X_valid),
            "test_rows_reserved":len(X_test)
        })
        mlflow.set_tags({
            "stage":"candidate", "course":"Kreo Digital Session 09",
            "data_source":"sklearn.make_classification synthetic",
            "data_fingerprint_sha256":dataset_fingerprint,
            "python_version":platform.python_version(), "sklearn_version":sklearn.__version__,
            "git_sha":os.getenv("GITHUB_SHA","unknown")
        })
        mlflow.log_metrics(metrics)
        mlflow.log_dict({"splits":counts, "metric_definition":"sklearn average_precision_score",
                         "threshold":0.5, "synthetic":True}, "reports/experiment_manifest.json")
        signature = infer_signature(X_valid[:10], estimator.predict(X_valid[:10]))
        mlflow.sklearn.log_model(
            sk_model=estimator, name="model", signature=signature,
            input_example=X_valid[:3],
            # MLflow 3.x saves sklearn models with skops; trust the tree type of our own RandomForest
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
        results.append({"model":model_name,"run_id":run.info.run_id,**metrics})
summary = pd.DataFrame(results).sort_values("validation_pr_auc",ascending=False)
display(summary)

## 4 — Select with validation, register, and alias
Select by validation PR-AUC. Registry aliases (such as `champion`) support controlled references without assuming a stage transition. Registration is **not** deployment. A real production promotion requires approvals, monitoring, and rollback readiness.

In [ ]:
best = summary.iloc[0]
best_run_id = str(best["run_id"])
model_uri = f"runs:/{best_run_id}/model"
registered_name = "kreo_session09_synthetic_risk"
registered = mlflow.register_model(model_uri=model_uri, name=registered_name)
client = MlflowClient()
client.set_registered_model_alias(registered_name, "candidate", registered.version)
version_uri = f"models:/{registered_name}@candidate"
loaded_model = mlflow.sklearn.load_model(version_uri)
assert np.array_equal(loaded_model.predict(X_valid[:20]), 
                      candidates[str(best["model"])].predict(X_valid[:20]))
print({"registered_model":registered_name, "version":registered.version,
       "alias":"candidate", "selected_validation_pr_auc":float(best["validation_pr_auc"])})

## 5 — One final holdout test
Do not use the test result to reselect a model. The purpose is unbiased reporting of the selected policy on a held-out dataset. No real-world fraud risk claim follows from synthetic data.

In [ ]:
test_scores = loaded_model.predict_proba(X_test)[:,1]
test_pred = (test_scores >= 0.5).astype(int)
test_metrics = {
    "test_pr_auc":float(average_precision_score(y_test,test_scores)),
    "test_roc_auc":float(roc_auc_score(y_test,test_scores)),
    "test_precision_at_05":float(precision_score(y_test,test_pred,zero_division=0)),
    "test_recall_at_05":float(recall_score(y_test,test_pred,zero_division=0)),
}
with mlflow.start_run(run_name="final-holdout-audit") as audit:
    mlflow.set_tags({"stage":"final-evaluation", "candidate_run_id":best_run_id,
                     "dataset":"synthetic", "selection_rule":"validation_pr_auc"})
    mlflow.log_metrics(test_metrics)
    mlflow.log_dict({"confusion_matrix":confusion_matrix(y_test,test_pred).tolist(),
                     "test_size":len(y_test),"model_uri":version_uri},
                    "reports/holdout_audit.json")
print(test_metrics)

## 6 — Inspect runs and compare experiments programmatically
Model selection must be reproducible from stored metadata, not just a screenshot of the tracking UI.

In [ ]:
experiment = mlflow.get_experiment_by_name(experiment_name)
stored = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.stage = 'candidate'",
    order_by=["metrics.validation_pr_auc DESC"],
)
display(stored[["run_id","tags.mlflow.runName","metrics.validation_pr_auc",
                "metrics.validation_roc_auc"]].head())
assert set(summary["run_id"]).issubset(set(stored["run_id"]))

## 7 — Deploy the registered model as a REST endpoint
Serve the **registry alias**, not a run ID. That way a rollback or promotion is just moving the alias and restarting the server, with no code change.

`mlflow models serve` loads `models:/kreo_session09_synthetic_risk@candidate` once at startup and exposes:
- `GET /ping`: health check
- `POST /invocations`: predictions (JSON body `{"inputs": [[...18 features...], ...]}`)

**Docker (recommended with the Compose stack):**
```bash
docker compose --profile serving up -d --build
# after re-registering or moving the alias, reload the model:
docker compose --profile serving restart model-server
```

**Without Docker** (in your activated venv, tracking to the same server):
```bash
MLFLOW_TRACKING_URI=http://127.0.0.1:5000 mlflow models serve \
  -m "models:/kreo_session09_synthetic_risk@candidate" --env-manager local --port 5001
```

> The endpoint returns class labels (threshold 0.5), because the logged pyfunc uses `predict`. Serving is not production readiness: there is no auth, TLS, autoscaling, request logging or drift monitoring here.

In [ ]:
import requests

SERVING_URL = os.getenv("MODEL_SERVING_URL", "http://127.0.0.1:5001")
try:
    ping = requests.get(f"{SERVING_URL}/ping", timeout=5)
    server_up = ping.status_code == 200
except requests.ConnectionError:
    server_up = False

if not server_up:
    print(f"No model server at {SERVING_URL}. Start it (see the markdown above), then re-run this cell.")
else:
    batch = X_test[:5]
    response = requests.post(f"{SERVING_URL}/invocations", json={"inputs": batch.tolist()}, timeout=30)
    response.raise_for_status()
    served = np.asarray(response.json()["predictions"])
    expected = loaded_model.predict(batch)
    # A mismatch usually means the alias moved after the server started: restart model-server.
    assert np.array_equal(served, expected), (served, expected)
    print({"endpoint": f"{SERVING_URL}/invocations",
           "served_predictions": served.tolist(),
           "matches_registry_model": True})

## 8 — Best-practice discussion: scaling and production boundaries
| Concern | Small lab | Team/prod design |
|---|---|---|
| Metadata | SQLite | Managed PostgreSQL with backups/migrations |
| Artifacts | Local disk | S3 / compatible object storage |
| API | One local process | Stateless replicas behind ingress/load balancer |
| Security | localhost only | TLS, authn/authz, least privilege, audit logging |
| Serving | Local model.load | Independent serving fleet and rollout controls |
| Operations | Manual run | CI checks, alerts, retention policies, recovery drills |

**Decision exercise:** Explain when you would choose direct artifact access versus MLflow's artifact proxy. What happens when PostgreSQL is unavailable? Which artifacts and metadata must be backed up together? How would you validate an upgrade and roll back a model alias?

**Caution:** MLflow tracking-server replicas scale request handling, not GPU training or inference. Object-store and DB bottlenecks remain independent.

## 9 — Student exercises (submit written answers)
1. Change the random seed and explain why it affects run comparability.
2. Add an ML model and log the hyperparameters meaningfully.
3. Change the decision threshold using validation only. Report precision/recall at a fixed review budget.
4. Produce a model card stating dataset origin, limitations, expected operational impact, and monitoring plan.
5. Propose separate development/staging/production model promotion gates.
6. Draw a scaled MLflow architecture with an ingress, 2+ replicas, PostgreSQL, and object storage.
7. Explain what happens when the metadata database fails but object storage is available.
8. Identify the information required to reproduce your best model in six months.
9. Explain why the synthetic data cannot establish production accuracy.
10. Design a backup-restore drill and a measurable SLO.

**Submission:** completed notebook, MLflow run IDs, validation vs holdout comparison, architecture diagram, and one-page governance proposal.

**Assessment:** experiment correctness 25%; reproducibility 20%; model registry 15%; scale/security architecture 25%; critical analysis 15%.

**References:** https://mlflow.org/docs/latest/